# BÁO CÁO THỰC HÀNH TUẦN 2 <br>
# Môn Học: Khai Thác Dữ liệu
``Họ và tên: Phạm Ngọc Hào`` <br>
``MSSV: 23110146``

Trong phần bài tập này, ta sẽ dụng hai hàm **Part1()** và **Par2()** để thực hiện các đề bài đã giao. Ngoài ra, trong quá trình thực thi các hàm tính toán các khoảng cách và độ đo, ta dùng kỹ thuật **Test-Driven Development**.

Để bắt đầu, ta cần sử dụng các thư viện cần thiết như sau:

In [1]:
import pandas as pd
import numpy as np
import unittest

# I. Khởi tạo và tiền xử lý dữ liệu

Để thuận tiện trong việc nhập dữ liệu, ta sẽ tạo một class `DataLoader` với nhiệm vụ tự động hóa nhập dữ liệu cần thiết cho bài tập này.

In [2]:
class DataLoader:
    def __init__(self):
        self.ionosphere_path = "./data/ionosphere.data"
        self.kddcup_path = "./data/kddcup.data.corrected"
    
    def ionosphere_load(self):
        data = pd.read_csv(self.ionosphere_path, header = None)
        return data
    
    def kddcup_load(self):
        cols = [1, 2, 3, 41]
        data = pd.read_csv(self.kddcup_path, header = None, usecols = cols)
        categorical_data = data.select_dtypes(include=['object'])
        return categorical_data.drop_duplicates().reset_index(drop=True)

Cụ thể, ta dùng `pandas` để đọc hai bộ dữ liệu cần thiếu được mô tả như sau:

1. Bộ dữ liệu Ionosphere (Dữ liệu số)

- Nguồn gốc: Kho Học máy UCI, chứa dữ liệu thu thập từ hệ thống radar phân loại tín hiệu tầng điện ly.

- Đặc điểm: Gồm 351 dòng (bản ghi) và 35 cột. Các đặc trưng từ cột 0 đến 33 là dữ liệu số liên tục (nằm trong khoảng -1 đến 1).

- Tiền xử lý & Sử dụng: Cột 34 (cột cuối chứa nhãn phân loại 'g' hoặc 'b') được loại bỏ để ma trận chỉ còn lại dữ liệu số. Dữ liệu này được dùng để tính toán các khoảng cách không gian (Manhattan, Euclide, Max Norm).

2. Bộ dữ liệu KDD Cup 1999 (Dữ liệu phân loại)

- Nguồn gốc: Kho Học máy UCI, là bộ dữ liệu tiêu chuẩn dùng trong bài toán phát hiện xâm nhập mạng (phân biệt kết nối bình thường và các loại tấn công).

- Đặc điểm: Dữ liệu gốc có kích thước rất lớn (hơn 4,8 triệu dòng) bao gồm cả đặc trưng số và chữ.

- Tiền xử lý & Sử dụng: Bài thực hành chỉ trích xuất 4 cột chứa dữ liệu phân loại (chuỗi ký tự) bao gồm: giao thức, dịch vụ, trạng thái (cột 1, 2, 3) và nhãn (cột 41). Sau khi loại bỏ các bản ghi trùng lặp, dữ liệu được thu gọn (còn 609 dòng) để phục vụ cho việc tính toán độ tương đồng bằng độ đo Overlap và Tần suất xuất hiện ngược.

# II. Yêu cầu 1: Tính các chuẩn khác nhau cho bộ dữ liệu ionosphere

## 1. Hàm tính chuẩn giữa hai điểm

Hàm `norm()` được xây dựng để tính chuẩn $L_{p}$ giữa một véc-tơ (chênh lệch của 2 điểm).

In [3]:
def norm(vector, p = 1):
    vector = np.asarray(vector)

    if p == "inf":
        return np.max(np.abs(vector))
    
    if p < 1:
        raise ValueError("p must be greater than or equal 1")

    else:
        return np.power(np.sum(np.power(np.abs(vector), p)), 1/p)

Để kiểm tra tính đúng đắn của hàm `norm()`, class `TestNorm` sẽ thực thi tính logic và cú pháp của hàm `norm`. 

In [4]:
class TestNorm(unittest.TestCase):
    def test_negative_p(self):
        with self.assertRaises(ValueError):
            norm([1, 2], p = 0)

    def test_l1(self):
        x = range(1, 7)
        l1 = norm(x, p = 1)
        self.assertAlmostEqual(l1, np.linalg.norm(x, ord = 1))

    def test_l2(self):
        x  = [3, 4]
        l2 = norm(x, p = 2)
        self.assertAlmostEqual(l2, np.linalg.norm(x, ord = 2))

    def test_linf(self):
        x = range(0, 15)
        linf = norm(x, p = "inf")
        self.assertEqual(linf, 14)

    def test_l10(self):
        x = range(0, 4)
        l10 = norm(x, p = 10)
        self.assertAlmostEqual(l10, np.linalg.norm(x, ord = 10))

## 2. Hàm tính chuẩn cho nhiều điểm

Để tính chuẩn $L_p$ cho nhiều điểm trong bộ dữ liệu, thay vì dùng vòng lặp cho mỗi dòng, ta sử dụng tính chất **broadcasting** của thư viện **NumPy**. 

In [5]:
def calculate_norm_batches(tensor, p = 1, batch_size = 50):
    tensor = tensor[:batch_size, :]
    diff = tensor[:, None, :] - tensor[None, :, :]
    if p == "inf":
        return np.max(np.abs(diff), axis = 2)
    else:
        return np.power(np.sum(np.power(np.abs(diff), p), axis = 2) , 1/p)

Ta tạo ra ma trận `diff = tensor[:, None, :] - tensor[None, :, :]` ba chiều để lưu trữ hiệu tọa độ của các cặp điểm $i$ và $j$ trong một lô dữ liệu (ở đây lô dữ liệu cần là 50). Sau đó, ta dùng công thức tính chuẩn $L_p$ cho lô dữ liệu được thực hiện dọc trung thứ ba `axis = 2`. Kết quả là một ma trận khoảng cách chứ khoảng cách tương ứng giữa cách điểm.

Tương tự, để kiểm tra tính đúng đắn của hàm vừa thực thi, ta sử dụng class `TestNormBatches` như dưới đây:

In [6]:
class TestNormBatches(unittest.TestCase):
    def setUp(self):
        self.data = np.array([
            [3, 1],
            [1, 2],
            [2, 0],
            [2, 3]
        ])

    def test_l1(self):
        result = calculate_norm_batches(self.data, p = 1, batch_size= 4)
        self.assertEqual(result[0, 1], 3)
        self.assertEqual(result[0, 2], 2)
        self.assertEqual(result[1, 0], 3)
        self.assertEqual(result[0, 0], 0)

    def test_l2_(self):
        result = calculate_norm_batches(self.data, p = 2, batch_size = 4)
        self.assertAlmostEqual(result[0, 1], np.sqrt(5))
        self.assertAlmostEqual(result[2, 3], 3.0)

    def test_linf(self):
        result = calculate_norm_batches(self.data, p = "inf", batch_size = 4)
        self.assertEqual(result[0, 1], 2)
        self.assertEqual(result[0, 2], 1)

    def test_output_shape(self):
        batches = 3
        result = calculate_norm_batches(self.data, p = 2, batch_size = batches)
        self.assertEqual(result.shape, (3, 3))

## 3. Hàm thực thi yêu cầu 1

In [7]:
def Part1():
    print("="*50)
    print("Part 1:")
    print("="*50)
    data_loader = DataLoader()

    print("# 1. Load the Ionosphere data")
    ionosphere_data = data_loader.ionosphere_load()
    print(ionosphere_data)
    print()

    print("# 2. Drop the end column")
    ionosphere_data.pop(ionosphere_data.columns[-1])
    print(ionosphere_data)
    print()

    print("# 3. Initialize the points point1, point2 " \
    "corresponding to lines 0, 1, and 2 of the array, "
    "and calculate the L1, L2, L_inf norms")
    ionosphere_array = ionosphere_data.values
    print(ionosphere_array)
    print()

    point1 = ionosphere_array[0, :]
    point2 = ionosphere_array[1, :]
    point3 = ionosphere_array[2, :]
    
    # p = 1
    dist1_p1_p2 = norm(point1 - point2, p = 1)
    dist1_p1_p3 = norm(point1 - point3, p = 1)

    # p = 2
    dist2_p1_p2 = norm(point1 - point2, p = 2)
    dist2_p1_p3 = norm(point1 - point3, p = 2)

    # p = inf
    dist_inf_p1_p2 = norm(point1 - point2, p = "inf")
    dist_inf_p1_p3 = norm(point1 - point3, p = "inf")

    print("L1 distance between point1 and point2:", dist1_p1_p2)
    print("L1 distance between point1 and point3:", dist1_p1_p3)

    print("L2 distance between point1 and point2:", dist2_p1_p2)
    print("L2 distance between point1 and point3:", dist2_p1_p3)

    print("L-infinity distance between point1 and point2:", dist_inf_p1_p2)
    print("L-infinity distance between point1 and point3:", dist_inf_p1_p3)

    print()

    print("# 4. Using function to calculate" \
    " the L1, L2, L_inf norm for 50 fisrt lines")
    print("The L1 norm distance-matrix" \
    " of the first 50 lines in the Ionosphere Data")
    L1_matrix_dist = calculate_norm_batches(
        ionosphere_array, p = 1, batch_size=50
    )
    print(pd.DataFrame(L1_matrix_dist))
    print()
    print("The L2 norm distance-matrix" \
    " of the first 50 lines in the Ionosphere Data")
    L2_matrix_dist = calculate_norm_batches(
        ionosphere_array, p = 2, batch_size=50
    )
    print(pd.DataFrame(L2_matrix_dist))
    print()
    print("The L_inf norm distance-matrix" \
    " of the first 50 lines in the Ionosphere Data")
    L_inf_matrix_dist = calculate_norm_batches(
        ionosphere_array, p = "inf", batch_size=50
    )
    print(pd.DataFrame(L_inf_matrix_dist))
    print()

# III. Yêu cầu 2: Tính các láng giềng gần nhất ở mục 2 sử dụng độ đo Overlap và độ đo tần suất xuất hiện ngược cho 100 dòng đầu tiên

## 1. Độ đo overlap

Độ đo Overlap so sánh từng cặp thuộc tính, nếu giống nhau tính là 1, khác nhau là 0, sau đó chia đều trọng số.

In [8]:
def overlap(df, batch_size = None):
    if batch_size is None:
        batch_size = len(df)

    subset = df.iloc[: batch_size, :3].values
    matches = (subset[:, None, :] == subset[None, :, :])
    return np.mean(matches, axis = 2)

Biến matches so sánh các dòng với nhau để tạo mảng True/False (1/0) dựa trên kỹ thuật broadcasting. Hàm np.mean(..., axis = 2) sẽ tính trung bình số lượng các thuộc tính trùng khớp. Do trọng số $w_{i} = 1/d$ (ở đây $d=3$), việc dùng trung bình cộng chính là cách tính $Sim(\overline{X}, \overline{Y})$ hoàn hảo và ngắn gọn

Tương tự, ta cũng có một class `TestOverlap` như dưới để kiểm tra tính đúng đắn của hàm `overlap` ở trên.

In [9]:
class TestOverlap(unittest.TestCase):
    def setUp(self):
        self.data = pd.DataFrame([
            ['tcp', 'http', 'SF'],
            ['tcp', 'smtp', 'SF'],
            ['udp', 'http', 'REJ'],
            ['udp', 'dns', 'S0']
        ])

    def test_identity(self):
        result = overlap(self.data)
        self.assertEqual(result[0, 0], 1.0)
        self.assertEqual(result[2, 2], 1.0)

    def test_partial_match(self):
        result = overlap(self.data)
        self.assertAlmostEqual(result[0, 1], 2/3)

    def test_low_match(self):
        result = overlap(self.data)
        self.assertAlmostEqual(result[0, 2], 1/3)

    def test_no_match(self):
        result = overlap(self.data)
        self.assertEqual(result[0, 3], 0.0)

    def test_symmetry(self):
        result = overlap(self.data)
        self.assertEqual(result[1, 2], result[2, 1])

    def test_output_shape(self):
        result = overlap(self.data, batch_size=3)
        self.assertEqual(result.shape, (3, 3))

## 2. Độ đo tần suất xuất hiện ngược (Inverse Frequency)

Độ đo này ưu tiên sự tương đồng ở các giá trị hiếm. Nếu hai bản ghi cùng sở hữu một giá trị xuất hiện ít trong tập dữ liệu thì độ tương đồng sẽ cao hơn.

In [10]:
def inverse_frequence(df, batch_size = 100):
    subset = df.iloc[: batch_size, :3]

    log_freq_maps = []
    for col in subset.columns:
        freqs = df[col].value_counts()
        log_freq_maps.append(np.log10(freqs).to_dict())

    log_f_matrix = np.array([
        [log_freq_maps[c][val] for c, val in enumerate(row)]
        for row in subset.values
    ])

    log_prod = log_f_matrix[:, None, :] * log_f_matrix[None, :, :]

    s_mismatch = 1 / (1 + log_prod)
    matches = (subset.values[:, None, :] == subset.values[None, :, :])

    sim_components = np.where(matches, 1.0, s_mismatch)
    return np.mean(sim_components, axis = 2)

Ta sử dụng value_counts() để đếm $f_i(x)$ và lưu logarit cơ số 10 vào từ điển log_freq_maps. Ta cũng dùng log_f_matrix thay thế ma trận chuỗi ban đầu bằng ma trận số chứa các giá trị log tương ứng. Và cuối cùng, ta dùng np.where(matches, 1.0, s_mismatch) là cơ chế rẽ nhánh: áp dụng logic gán bằng 1 nếu hai thuộc tính giống hệt nhau, và gán bằng công thức đảo ngược logarit nếu khác nhau. Cuối cùng tính trung bình cộng bằng np.mean.

Ta cũng có một class `TestInverseFrequency` để kiểm tra tính đúng đắn của `inverse_frequence` như dưới đây:

In [11]:
class TestInverseFrequency(unittest.TestCase):
    def setUp(self):
        data = []
        for _ in range(100): data.append(['A', 'X', 'Y'])
        for _ in range(10):  data.append(['B', 'X', 'Y'])
        
        self.df = pd.DataFrame(data)

    def test_identity(self):
        result = inverse_frequence(self.df, batch_size=5)
        self.assertAlmostEqual(result[0, 0], 1.0)

    def test_symmetry(self):
        result = inverse_frequence(self.df, batch_size=110)
        self.assertAlmostEqual(result[0, 100], result[100, 0])

    def test_math_logic(self):
        result = inverse_frequence(self.df, batch_size=110)
        
        expected_sim = ((1/(1 + 2*1)) + 1.0 + 1.0) / 3
        self.assertAlmostEqual(result[0, 100], expected_sim, places=4)

    def test_rare_values(self):
        rare_df = pd.concat(
            [self.df, pd.DataFrame([['C', 'X', 'Y']])], ignore_index=True
        )
        result = inverse_frequence(rare_df, batch_size=111)
        
        self.assertAlmostEqual(result[0, 110], 1.0)

## 3. Tìm láng giềng gần nhất

In [12]:
def nearest_neighbors(sim_matrix):
    return np.argsort(-sim_matrix, axis = 1)

Hàm np.argsort trả về chỉ mục của mảng sau khi sắp xếp. Bằng cách truyền vào -sim_matrix (thêm dấu âm), mảng sẽ được sắp xếp theo độ tương đồng giảm dần (lớn nhất đứng đầu), qua đó tìm được K-láng giềng có độ tương đồng cao nhất.

## 4. Hàm thực thi yêu cầu 2

In [13]:
def Part2():
    print("="*50)
    print("Part 2:")
    print("="*50)
    data_loader = DataLoader()

    kddcup_data = data_loader.kddcup_load()
    print("# 1.  Load the KDD Cup data")
    print(kddcup_data.head(10))
    print()

    print("# 2. Find the nearest neighbors using Overlap metric ")
    sim_matrix_overlap = overlap(kddcup_data, batch_size = 100)
    nn_overlap = nearest_neighbors(sim_matrix = sim_matrix_overlap)
    print(nn_overlap)
    print()

    print("# 3. Find the nearest neighbors using Inverse Frequency")
    sim_matrix_if = inverse_frequence(kddcup_data, batch_size = 100)
    nn_if = nearest_neighbors(sim_matrix = sim_matrix_if)
    print(nn_if)
    print()

# IV.Kiểm thử và thực thi chương trình

## 1. Kiểm thử các hàm tính toán

Xuyên suốt file week02.py, phương pháp TDD (Test-Driven Development) đã được áp dụng. Các lớp TestNorm, TestNormBatches, TestOverlapSimilarity, TestInverseFrequency kế thừa từ unittest.TestCase liên tục gọi dữ liệu mẫu để kiểm chứng các hàm.

In [14]:
unittest.main(argv = [''], verbosity = 2, exit = False)

test_identity (__main__.TestInverseFrequency) ... ok
test_math_logic (__main__.TestInverseFrequency) ... ok
test_rare_values (__main__.TestInverseFrequency) ... ok
test_symmetry (__main__.TestInverseFrequency) ... ok
test_l1 (__main__.TestNorm) ... ok
test_l10 (__main__.TestNorm) ... ok
test_l2 (__main__.TestNorm) ... ok
test_linf (__main__.TestNorm) ... ok
test_negative_p (__main__.TestNorm) ... ok
test_l1 (__main__.TestNormBatches) ... ok
test_l2_ (__main__.TestNormBatches) ... ok
test_linf (__main__.TestNormBatches) ... ok
test_output_shape (__main__.TestNormBatches) ... ok
test_identity (__main__.TestOverlap) ... ok
test_low_match (__main__.TestOverlap) ... ok
test_no_match (__main__.TestOverlap) ... ok
test_output_shape (__main__.TestOverlap) ... ok
test_partial_match (__main__.TestOverlap) ... ok
test_symmetry (__main__.TestOverlap) ... ok

----------------------------------------------------------------------
Ran 19 tests in 0.075s

OK


Các hàm tính toán đều vượt qua các test cases. Điều này nghĩa là các hàm hoạt động tốt.

## 2. Thực thi chương trình

In [15]:
def main():
    # unittest.main(argv = [''], verbosity = 2, exit = False)
    Part1()

    Part2()

In [16]:
if __name__ == "__main__":
    main()

Part 1:
# 1. Load the Ionosphere data
     0   1        2        3        4        5        6        7        8   \
0     1   0  0.99539 -0.05889  0.85243  0.02306  0.83398 -0.37708  1.00000   
1     1   0  1.00000 -0.18829  0.93035 -0.36156 -0.10868 -0.93597  1.00000   
2     1   0  1.00000 -0.03365  1.00000  0.00485  1.00000 -0.12062  0.88965   
3     1   0  1.00000 -0.45161  1.00000  1.00000  0.71216 -1.00000  0.00000   
4     1   0  1.00000 -0.02401  0.94140  0.06531  0.92106 -0.23255  0.77152   
..   ..  ..      ...      ...      ...      ...      ...      ...      ...   
346   1   0  0.83508  0.08298  0.73739 -0.14706  0.84349 -0.05567  0.90441   
347   1   0  0.95113  0.00419  0.95183 -0.02723  0.93438 -0.01920  0.94590   
348   1   0  0.94701 -0.00034  0.93207 -0.03227  0.95177 -0.03431  0.95584   
349   1   0  0.90608 -0.01657  0.98122 -0.01989  0.95691 -0.03646  0.85746   
350   1   0  0.84710  0.13533  0.73638 -0.06151  0.87873  0.08260  0.88928   

          9   ...       2

# V. Kết luận

Qua bài thực hành tuần 2, các mục tiêu đề ra đã được hoàn thành đầy đủ cả về mặt lý thuyết: Hiểu và phân biệt rõ phương pháp đo lường cho từng loại dữ liệu. Đối với dữ liệu số (Ionosphere), khoảng cách không gian giữa các điểm được đo lường chính xác bằng các chuẩn Minkowski ($L_1$, $L_2$, $L_\infty$). Đối với dữ liệu phân loại (KDD Cup), mức độ giống nhau được đánh giá thông qua độ đo Overlap (trùng khớp trực tiếp) và Tần suất xuất hiện ngược (đánh giá cao sự trùng khớp ở các đặc trưng hiếm).